# LAB: DECISION TREE CLASSIFICATION

**Dataset:** `Iris.csv` — 150 samples, 3 species (*Iris-setosa*, *Iris-versicolor*, *Iris-virginica*)
**Features:** `SepalLengthCm`, `SepalWidthCm`, `PetalLengthCm`, `PetalWidthCm`  
**Target:** `Species` (3 classes)

---
### Objectives
- Implement core Decision Tree components from scratch: Entropy, Gini Impurity, Information Gain, Recursive Tree Construction.
- Apply Scikit-Learn's `DecisionTreeClassifier` with hyperparameter tuning and cross-validation.
- Visualise decision boundaries and tree structure on a real multiclass dataset.

## PART 1: IMPLEMENTING DECISION TREE FROM SCRATCH

### 1.1. Theoretical Background

A Decision Tree partitions the feature space by selecting at each node the split $(j,\,t)$
that maximises **Information Gain (IG)**:

$$\text{IG}(y,\,j,\,t) = \text{Impurity}(y) - \frac{N_L}{N}\,\text{Impurity}(y_L) - \frac{N_R}{N}\,\text{Impurity}(y_R)$$

Two common impurity metrics:

| Criterion | Formula |
|-----------|---------|
| **Shannon Entropy** | $H(y)=-\displaystyle\sum_c p_c\log_2 p_c$ |
| **Gini Impurity**   | $G(y)=1-\displaystyle\sum_c p_c^2$ |

**Splitting rule:** samples with $x_j \le t$ go *left*; samples with $x_j > t$ go *right*.
The tree stops growing when `max_depth` is reached, a node has fewer than `min_samples_split` samples,
or the node is already pure (all labels identical).

### 1.2. Impurity Metrics
Complete the `# TODO` blocks. Each blank is marked with `# Fill in your code here`.

In [ ]:
import numpy as np

def calculate_entropy(y):
    """
    Compute Shannon Entropy: H(y) = -sum_c p_c * log2(p_c).

    Parameters
    ----------
    y : np.ndarray, shape (n_samples,) — integer class labels

    Returns
    -------
    float : entropy value (0 = pure node, 1 = maximally mixed for binary)
    """
    if len(y) == 0:
        return 0.0

    # TODO: Count occurrences of each class label
    counts = np.bincount(y)  # Fill in your code here  Hint: np.bincount(y)

    # TODO: Convert counts to probabilities
    probabilities = counts / len(y)  # Fill in your code here

    # TODO: Compute entropy (add 1e-9 inside log to avoid log(0))
    entropy = -np.sum(probabilities * np.log2(probabilities + 1e-9))  # Fill in your code here
    return entropy


def calculate_gini(y):
    """
    Compute Gini Impurity: G(y) = 1 - sum_c p_c^2.

    Parameters
    ----------
    y : np.ndarray, shape (n_samples,) — integer class labels

    Returns
    -------
    float : Gini value (0 = pure, 0.5 = maximally mixed for binary)
    """
    if len(y) == 0:
        return 0.0

    # TODO: Count occurrences of each class label
    counts = np.bincount(y)  # Fill in your code here

    # TODO: Convert counts to probabilities
    probabilities = counts / len(y)  # Fill in your code here

    # TODO: Compute Gini impurity
    gini = 1.0 - np.sum(probabilities ** 2)  # Fill in your code here
    return gini


# --- Sanity check ---
sample_y = np.array([0, 0, 1, 1, 2, 2])   # balanced 3-class
print(f"Entropy (balanced 3-class) : {calculate_entropy(sample_y):.4f}  (expected ~1.5850)")
print(f"Gini    (balanced 3-class) : {calculate_gini(sample_y):.4f}  (expected ~0.6667)")
print(f"Entropy (pure node)        : {calculate_entropy(np.array([1,1,1])):.4f}  (expected 0.0000)")
print(f"Gini    (pure node)        : {calculate_gini(np.array([1,1,1])):.4f}  (expected 0.0000)")

### 1.3. Dataset Splitting and Information Gain

In [ ]:
def split_dataset(X_column, threshold):
    """
    Partition one feature column into left and right index sets.

    Left  : indices where X_column <= threshold
    Right : indices where X_column >  threshold

    Parameters
    ----------
    X_column  : np.ndarray, shape (n_samples,)
    threshold : float

    Returns
    -------
    left_idxs, right_idxs : np.ndarray of int
    """
    # TODO: Return the left and right index arrays
    left_idxs  = np.argwhere(X_column <= threshold).flatten()  # Fill in your code here  Hint: np.argwhere(X_column <= threshold).flatten()
    right_idxs = np.argwhere(X_column >  threshold).flatten()  # Fill in your code here
    return left_idxs, right_idxs


def calculate_information_gain(y, left_idxs, right_idxs, criterion='gini'):
    """
    IG = Impurity(parent) - (N_L/N)*Impurity(left) - (N_R/N)*Impurity(right)

    Parameters
    ----------
    y          : np.ndarray — full label array before the split
    left_idxs  : np.ndarray — row indices routed to the left child
    right_idxs : np.ndarray — row indices routed to the right child
    criterion  : str — 'gini' or 'entropy'

    Returns
    -------
    float : information gain (>= 0)
    """
    impurity_fn = calculate_gini if criterion == 'gini' else calculate_entropy
    n = len(y)

    # TODO: Compute parent node impurity
    parent_impurity = impurity_fn(y)  # Fill in your code here

    # TODO: Compute weighted child impurity
    #       Hint: (len(left_idxs)/n)*impurity_fn(y[left_idxs]) + ...
    weighted_child = ((len(left_idxs) / n) * impurity_fn(y[left_idxs])
                      + (len(right_idxs) / n) * impurity_fn(y[right_idxs]))  # Fill in your code here

    # TODO: Return information gain
    return parent_impurity - weighted_child  # Fill in your code here

### 1.4. Finding the Best Split and Building the Decision Tree

In [ ]:
def find_best_split(X, y, criterion='gini'):
    """
    Search every feature and candidate threshold to find the split
    yielding the maximum Information Gain.

    Parameters
    ----------
    X         : np.ndarray, shape (n_samples, n_features)
    y         : np.ndarray, shape (n_samples,)
    criterion : str — 'gini' or 'entropy'

    Returns
    -------
    best_feature   : int   — column index of best feature
    best_threshold : float — optimal threshold value
    best_gain      : float — information gain of the best split
    """
    best_feature, best_threshold, best_gain = None, None, -np.inf
    n_features = X.shape[1]

    for feat in range(n_features):
        thresholds = np.unique(X[:, feat])
        for thresh in thresholds:
            # TODO: Split dataset on (feat, thresh)
            left_idxs, right_idxs = split_dataset(X[:, feat], thresh)  # Fill in your code here

            if len(left_idxs) == 0 or len(right_idxs) == 0:
                continue

            # TODO: Compute information gain for this split
            gain = calculate_information_gain(y, left_idxs, right_idxs, criterion)  # Fill in your code here

            # TODO: Update best split if this gain is higher
            if gain > best_gain:  # Fill in your code here
                best_feature, best_threshold, best_gain = feat, thresh, gain

    return best_feature, best_threshold, best_gain


# ── Node and Tree ────────────────────────────────────────────────────────

class Node:
    """Represents an internal or leaf node of a Decision Tree."""
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature   = feature    # Feature index used for splitting
        self.threshold = threshold  # Threshold value for the split
        self.left      = left       # Left subtree  (X[:, feature] <= threshold)
        self.right     = right      # Right subtree (X[:, feature] >  threshold)
        self.value     = value      # Majority class label at leaf nodes

    def is_leaf_node(self):
        return self.value is not None


class DecisionTreeScratch:
    """
    Decision Tree classifier implemented from scratch.
    Supports 'gini' and 'entropy' impurity criteria.
    """
    def __init__(self, min_samples_split=2, max_depth=5, criterion='gini'):
        self.min_samples_split = min_samples_split
        self.max_depth         = max_depth
        self.criterion         = criterion
        self.root              = None

    # ── Training ─────────────────────────────────────────────────────────
    def fit(self, X, y):
        """Build the decision tree by recursive partitioning."""
        # TODO: Initiate recursive tree building; assign result to self.root
        self.root = self._build_tree(X, y, depth=0)  # Fill in your code here

    def _build_tree(self, X, y, depth=0):
        """Recursively build and return a Node."""
        n_samples = X.shape[0]
        n_labels  = len(np.unique(y))

        # --- Stopping criteria ---
        if depth >= self.max_depth or n_samples < self.min_samples_split or n_labels == 1:
            # TODO: Compute the majority class label for this leaf
            leaf_value = np.bincount(y).argmax()  # Fill in your code here  Hint: np.bincount(y).argmax()
            return Node(value=leaf_value)

        # --- Find the best split ---
        feat, thresh, gain = find_best_split(X, y, self.criterion)
        if feat is None:
            return Node(value=np.bincount(y).argmax())

        # --- Split data ---
        left_idxs, right_idxs = split_dataset(X[:, feat], thresh)

        # TODO: Recursively build left subtree
        left_child  = self._build_tree(X[left_idxs], y[left_idxs], depth + 1)  # Fill in your code here

        # TODO: Recursively build right subtree
        right_child = self._build_tree(X[right_idxs], y[right_idxs], depth + 1)  # Fill in your code here

        return Node(feature=feat, threshold=thresh, left=left_child, right=right_child)

    # ── Prediction ───────────────────────────────────────────────────────
    def _traverse(self, x, node):
        """Traverse tree for a single sample x; return predicted class."""
        if node.is_leaf_node():
            return node.value
        if x[node.feature] <= node.threshold:
            # TODO: Recurse into the left subtree
            return self._traverse(x, node.left)  # Fill in your code here
        # TODO: Recurse into the right subtree
        return self._traverse(x, node.right)  # Fill in your code here

    def predict(self, X):
        """Return predicted class labels for all samples in X."""
        # TODO: Apply _traverse to every row of X and return as np.array
        return np.array([self._traverse(x, self.root) for x in X])  # Fill in your code here

### 1.5. Loading the Iris Dataset
We load `Iris.csv`, encode the target label, and prepare train/test splits to evaluate `DecisionTreeScratch`.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# --- Load dataset ---
df = pd.read_csv('Iris.csv')
df.drop(columns=['Id'], inplace=True)   # Id column is not a feature

# --- Encode target ---
le = LabelEncoder()
df['Species'] = le.fit_transform(df['Species'])  # 0=setosa, 1=versicolor, 2=virginica

X = df.drop(columns=['Species']).values
y = df['Species'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Classes :', le.classes_)
print(f"Training samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")
print(f"Features         : {X_train.shape[1]}")

### 1.6. Training DecisionTreeScratch and Comparing with Scikit-Learn
Once you have completed all `# TODO` blocks above, run the cell below to compare
your implementation against Scikit-Learn's `DecisionTreeClassifier`.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# --- Custom implementation ---
dt_scratch = DecisionTreeScratch(max_depth=5, criterion='gini')
dt_scratch.fit(X_train, y_train)
y_pred_scratch = dt_scratch.predict(X_test)

# --- Scikit-Learn baseline ---
dt_sklearn = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)
dt_sklearn.fit(X_train, y_train)
y_pred_sklearn = dt_sklearn.predict(X_test)

acc_scratch = accuracy_score(y_test, y_pred_scratch)
acc_sklearn = accuracy_score(y_test, y_pred_sklearn)

print('=== ACCURACY COMPARISON ON IRIS TEST SET ===')
print(f"DecisionTreeScratch         : {acc_scratch:.4f}")
print(f"DecisionTreeClassifier (sk) : {acc_sklearn:.4f}")

**Note**: A correct implementation should achieve accuracy comparable to Scikit-Learn,
confirming that recursive Information Gain maximisation correctly partitions the 4D Iris feature space.

### 1.7. Visualising the Decision Boundary
We project onto the two most informative features (`PetalLengthCm`, `PetalWidthCm`) to draw
2D decision boundaries for both implementations side by side.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

FEATURES  = ['PetalLengthCm', 'PetalWidthCm']
FEAT_IDX  = [2, 3]   # indices in the original feature matrix
COLORS    = ['#4e79a7', '#f28e2b', '#59a14f']
CMAP      = mcolors.ListedColormap(COLORS)

X2_train  = X_train[:, FEAT_IDX]
X2_test   = X_test[:,  FEAT_IDX]

# Re-train on 2-feature subsets
dt_scratch_2d = DecisionTreeScratch(max_depth=5, criterion='gini')
dt_scratch_2d.fit(X2_train, y_train)

dt_sklearn_2d = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)
dt_sklearn_2d.fit(X2_train, y_train)

def plot_boundary(ax, model, X, y, title):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.3, X[:, 0].max() + 0.3
    y_min, y_max = X[:, 1].min() - 0.3, X[:, 1].max() + 0.3
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = np.array(model.predict(np.c_[xx.ravel(), yy.ravel()])).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap=CMAP)
    for cls, col, lbl in zip([0,1,2], COLORS, le.classes_):
        mask = y == cls
        ax.scatter(X[mask, 0], X[mask, 1], c=col, edgecolors='k', s=40, label=lbl)
    ax.set_xlabel(FEATURES[0])
    ax.set_ylabel(FEATURES[1])
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, linestyle=':', alpha=0.4)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
try:
    plot_boundary(axes[0], dt_scratch_2d, X2_test, y_test,
                 'Decision Boundary — DecisionTreeScratch')
except Exception as e:
    axes[0].set_title('Complete implementation first')
    axes[0].text(0.5, 0.5, str(e), transform=axes[0].transAxes, ha='center', fontsize=8)

plot_boundary(axes[1], dt_sklearn_2d, X2_test, y_test,
             'Decision Boundary — Scikit-Learn')
plt.suptitle('Iris Classification — Petal Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## PART 2: USING SCIKIT-LEARN ON THE IRIS DATASET

In this section we apply Scikit-Learn's `DecisionTreeClassifier` to `Iris.csv`,
perform full EDA, preprocessing, hyperparameter tuning, and evaluation.

### 2.1. Exploratory Data Analysis (EDA)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df_raw = pd.read_csv('Iris.csv')
df_raw.head(10)

In [ ]:
print('Column names:', df_raw.columns.tolist())
print(f"Shape: {df_raw.shape}")

In [ ]:
df_raw.describe().round(2)

**Note**: All four morphological features are continuous and measured in centimetres.
Petal dimensions (`PetalLengthCm`, `PetalWidthCm`) tend to have higher variance across species
and are expected to carry the most discriminative power.

In [ ]:
print('Missing values per column:')
print(df_raw.isnull().sum())

**Note**: The Iris dataset contains no missing values — no imputation step is required.

In [ ]:
df_raw.dtypes

Check distribution of the target label `Species`.

In [ ]:
print('Class counts:')
print(df_raw['Species'].value_counts())
print()
print('Class proportions (%):')
print((df_raw['Species'].value_counts(normalize=True) * 100).round(2))

**Note**: The dataset is perfectly **balanced** — 50 samples per class (33.33% each).
Accuracy is a reliable primary metric here; no class-weighting adjustment is needed.

Visualise pairwise feature distributions coloured by species.

In [ ]:
sns.pairplot(df_raw.drop(columns=['Id']), hue='Species',
             palette={'Iris-setosa':'#4e79a7',
                      'Iris-versicolor':'#f28e2b',
                      'Iris-virginica':'#59a14f'},
             diag_kind='kde', plot_kws={'alpha': 0.7})
plt.suptitle('Pairwise Feature Distributions — Iris Dataset', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

**Note**: `PetalLengthCm` and `PetalWidthCm` provide near-perfect linear separation between
*Iris-setosa* and the other two species, making them the most informative features for a Decision Tree.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 5))
features = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']
palette  = {'Iris-setosa':'#4e79a7', 'Iris-versicolor':'#f28e2b', 'Iris-virginica':'#59a14f'}
for ax, feat in zip(axes, features):
    sns.boxplot(data=df_raw, x='Species', y=feat, palette=palette, ax=ax)
    ax.set_title(feat)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)
plt.suptitle('Feature Distribution per Species', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Note (Data Leakage):** The Exploratory Data Analysis (EDA) above is performed on the entire dataset for educational purposes to provide a complete overview. In a rigorous machine learning workflow, EDA and any decisions derived from it (such as feature selection) should be conducted **only on the training set** to prevent data leakage and ensure unbiased evaluation on the test set.

### 2.2. Data Preprocessing
- Drop the `Id` column (non-informative surrogate key).
- Encode the target label `Species` to integer codes.
- Split into Train / Test sets (80/20) with stratification.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df = df_raw.drop(columns=['Id'])

le = LabelEncoder()
y_enc = le.fit_transform(df['Species'])   # 0=setosa, 1=versicolor, 2=virginica
X_all = df.drop(columns=['Species']).values

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)
print(f"Training samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")
print(f"Features         : {X_train.shape[1]}")
print('Classes          :', le.classes_)

### 2.3. Training Decision Tree Classifier (Criterion Comparison)

#### 2.3.1. Gini Impurity Criterion

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

dt_gini = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
dt_gini.fit(X_train, y_train)
y_pred_gini = dt_gini.predict(X_test)

print(f"Accuracy (Gini): {accuracy_score(y_test, y_pred_gini):.4f}")
print(classification_report(y_test, y_pred_gini, target_names=le.classes_))

#### 2.3.2. Entropy (Information Gain) Criterion

In [ ]:
dt_entropy = DecisionTreeClassifier(criterion='entropy', max_depth=4, random_state=42)
dt_entropy.fit(X_train, y_train)
y_pred_entropy = dt_entropy.predict(X_test)

print(f"Accuracy (Entropy): {accuracy_score(y_test, y_pred_entropy):.4f}")
print(classification_report(y_test, y_pred_entropy, target_names=le.classes_))

**Note**: On the balanced Iris dataset both criteria typically reach 90–100% accuracy.
Any difference reflects subtle sensitivity to tie-breaking when multiple features yield equal Information Gain.

### 2.4. Cross-Validation and Hyperparameter Tuning

#### a) Tuning max_depth via 5-Fold Cross-Validation
We scan `max_depth` from 1 to 10 and record mean 5-fold accuracy.

In [ ]:
from sklearn.model_selection import cross_val_score

depth_range  = range(1, 11)
cv_mean      = []
cv_std       = []

for d in depth_range:
    dt     = DecisionTreeClassifier(max_depth=d, random_state=42)
    scores = cross_val_score(dt, X_train, y_train, cv=5, scoring='accuracy')
    cv_mean.append(scores.mean())
    cv_std.append(scores.std())

best_depth = list(depth_range)[np.argmax(cv_mean)]
print(f"Best max_depth : {best_depth}  |  CV Accuracy : {max(cv_mean):.4f}")

plt.figure(figsize=(8, 4))
plt.plot(depth_range, cv_mean, marker='o', linewidth=2, label='Mean CV Accuracy')
plt.fill_between(depth_range,
                 [m - s for m, s in zip(cv_mean, cv_std)],
                 [m + s for m, s in zip(cv_mean, cv_std)],
                 alpha=0.15, label='± 1 std')
plt.axvline(best_depth, color='red', linestyle='--', label=f'Best depth = {best_depth}')
plt.xlabel('max_depth')
plt.ylabel('5-Fold CV Accuracy')
plt.title('Validation Curve: Accuracy vs max_depth')
plt.legend()
plt.tight_layout()
plt.show()

**Note**: On Iris, accuracy typically plateaus at `max_depth = 3`.
Deeper trees memorise noise in the training set without improving generalisation.

#### b) Tuning min_samples_split via 5-Fold Cross-Validation

In [ ]:
split_range     = range(2, 31, 2)
cv_split_mean   = []

for s in split_range:
    dt     = DecisionTreeClassifier(max_depth=best_depth, min_samples_split=s, random_state=42)
    scores = cross_val_score(dt, X_train, y_train, cv=5, scoring='accuracy')
    cv_split_mean.append(scores.mean())

best_split = list(split_range)[np.argmax(cv_split_mean)]
print(f"Best min_samples_split : {best_split}  |  CV Accuracy : {max(cv_split_mean):.4f}")

plt.figure(figsize=(8, 4))
plt.plot(list(split_range), cv_split_mean, marker='s', color='#f28e2b', linewidth=2)
plt.axvline(best_split, color='red', linestyle='--', label=f'Best = {best_split}')
plt.xlabel('min_samples_split')
plt.ylabel('5-Fold CV Accuracy')
plt.title('Validation Curve: Accuracy vs min_samples_split')
plt.legend()
plt.tight_layout()
plt.show()

**Note**: `min_samples_split` has limited impact on Iris because the dataset is small and clean.
On noisier real-world datasets this parameter is critical for preventing overfitting.

**Note:** The manual tuning in this section is meant to illustrate the individual impact of hyperparameters like `max_depth` and `min_samples_split`. Since tuning them one by one can be redundant and miss optimal combinations, a joint parameter search like `GridSearchCV` (shown in the next section) is the recommended approach for official hyperparameter tuning.

### 2.5. Joint Hyperparameter Optimisation using GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'criterion'        : ['gini', 'entropy'],
    'max_depth'        : [2, 3, 4, 5, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

print('Best parameters  :', grid_search.best_params_)
print(f"Best CV Accuracy : {grid_search.best_score_:.4f}")

### 2.6. Evaluating the Best Model on the Independent Test Set

In [ ]:
from sklearn.metrics import (
    accuracy_score, classification_report,
    ConfusionMatrixDisplay
)

best_dt = grid_search.best_estimator_
y_pred  = best_dt.predict(X_test)

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=le.classes_))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=le.classes_,
    cmap='Blues', ax=ax
)
ax.set_title('Confusion Matrix — Best Decision Tree')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importances
FEATURE_NAMES = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']
importances   = best_dt.feature_importances_
feat_series   = pd.Series(importances, index=FEATURE_NAMES).sort_values(ascending=True)

plt.figure(figsize=(7, 4))
feat_series.plot(kind='barh', color='#4e79a7', edgecolor='white')
plt.xlabel('Importance Score')
plt.title('Feature Importances — Best Decision Tree')
plt.tight_layout()
plt.show()

print('Feature Importances:')
print(feat_series.sort_values(ascending=False).to_string())

**Note**: `PetalLengthCm` and `PetalWidthCm` typically dominate feature importance,
corroborating the pairplot observation that petal dimensions are the most discriminative features.

### 2.7. Visualising the Decision Tree Structure
Plot the tree learned by the best estimator to interpret its decision rules.

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(14, 6))
plot_tree(
    best_dt,
    feature_names=FEATURE_NAMES,
    class_names=le.classes_,
    filled=True,
    rounded=True,
    fontsize=9,
)
plt.title('Learned Decision Tree — Iris Dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Note**: The root node typically splits on `PetalLengthCm <= 2.45`, cleanly isolating
*Iris-setosa* in the left branch. Subsequent splits on petal width further distinguish
*Iris-versicolor* from *Iris-virginica*.